# MentalBERT baseline — Google Colab runner

Fine-tunes the naive multi-class MentalBERT baseline (`src/modeling/train.py`) and
produces held-out softmax predictions (`src/modeling/predict.py`).

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type →
Hardware accelerator → GPU (T4 is fine)*.

**One-time Drive setup.** Put your data and outputs on Google Drive so they survive
runtime resets. Create this layout in *MyDrive* (upload your local `Datasets/` into it):
```
MyDrive/mental-health-fyp/
    Datasets/            <- upload your local Datasets/ here (DAIC-WOZ/, RedditMentalHealth/)
    Models/              <- created automatically for checkpoints/splits/predictions
```
The same `src/modeling` code runs here and locally — only the `--data-root` /
`--artifacts-root` paths differ.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this if you named your Drive folder differently.
DRIVE_ROOT = '/content/drive/MyDrive/mental-health-fyp'
DATA_ROOT = f'{DRIVE_ROOT}/Datasets'
ARTIFACTS_ROOT = f'{DRIVE_ROOT}/Models'

import os
assert os.path.isdir(DATA_ROOT), f'Datasets not found at {DATA_ROOT} — upload them to Drive first.'
print('Datasets found:', os.listdir(DATA_ROOT))

## 3. Clone the repo

Clones the code into the (ephemeral) Colab runtime. Re-running this cell in a fresh
session re-clones; the *data and models* live on Drive, so nothing is lost.

In [ ]:
%cd /content
![ -d mh-repo ] && rm -rf mh-repo
!git clone https://github.com/SoshanW/mental-health-screening-fyp.git mh-repo
%cd /content/mh-repo

## 4. Install dependencies

Colab ships torch (CUDA) preinstalled, so we install only the rest of the modeling stack.

In [ ]:
# Colab ships torch (CUDA) preinstalled, so we install only the rest of the stack.
# transformers is pinned to the exact version the code is tested against locally
# (5.14.1), so the base_model/pooler_output behaviour is identical here and locally.
!pip install -q 'transformers==5.14.1' 'accelerate>=0.26.0' 'scikit-learn>=1.4'
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| CUDA available:', torch.cuda.is_available())

## 4b. Authenticate with Hugging Face (required — MentalBERT is gated)

`mental/mental-bert-base-uncased` is a **gated** repo, so downloading it needs a logged-in
token that has accepted the model terms. One-time: sign in to Hugging Face, open the
[model page](https://huggingface.co/mental/mental-bert-base-uncased) and click **Agree and
access repository**, then make a **read** token. Store it as a Colab Secret named
`HF_TOKEN` (key icon in the left sidebar → toggle notebook access) so the cell below logs
in automatically; otherwise it prompts you to paste the token. Re-run this cell after any
runtime reset.

In [ ]:
# MentalBERT (mental/mental-bert-base-uncased) is a GATED HF repo. One-time setup:
#   1. Sign in at huggingface.co.
#   2. Open https://huggingface.co/mental/mental-bert-base-uncased and click
#      "Agree and access repository".
#   3. Make a READ token at https://huggingface.co/settings/tokens.
# Recommended: store the token in Colab Secrets (key icon, left sidebar) as HF_TOKEN
# and enable notebook access -- then this cell logs in automatically every runtime.
# Otherwise the login() widget prompts you to paste it.
import os
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

if token:
    login(token=token)
    print('Authenticated to Hugging Face via stored token.')
else:
    login()  # opens a widget; paste your READ token

# Sanity check: confirm this account can actually see the gated MentalBERT repo.
from huggingface_hub import HfApi
HfApi().model_info('mental/mental-bert-base-uncased')
print('Access to mental/mental-bert-base-uncased confirmed.')

## 5. Sanity-check the harmonized data load

Optional but recommended — confirms the loaders see your Drive data before training.

In [ ]:
!python -m src.data --data-root "$DATA_ROOT"

## 6. (Optional) Fast wiring smoke-test

Runs the full train→predict pipeline in seconds on a tiny public model and 50 samples —
verifies the plumbing before committing to a real MentalBERT run. Skip once you trust it.

In [ ]:
!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "/content/smoke_models" \
    --model-name hf-internal-testing/tiny-random-bert \
    --max-train-samples 50 --num-epochs 1
!python -m src.modeling.predict --artifacts-root "/content/smoke_models"

In [ ]:
# Fast first run: 1 epoch, isolated on its own Drive dir so it never collides with
# (or gets resumed into) the full 3-epoch run below. Train -> predict -> confusion
# matrix, end to end on the REAL MentalBERT. ~1 h on a T4.
# If you hit CUDA out-of-memory at --batch-size 32, drop it to 16.
FAST_ROOT = f'{DRIVE_ROOT}/Models_fast'

!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "$FAST_ROOT" \
    --num-epochs 1 --batch-size 32 --fp16 --save-steps 500 --resume
!python -m src.modeling.predict --artifacts-root "$FAST_ROOT"

import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

pred = pd.read_csv(f'{FAST_ROOT}/predictions/test_predictions.csv')
labels = sorted(set(pred['true_condition']) | set(pred['predicted_condition']))
cm = confusion_matrix(pred['true_condition'], pred['predicted_condition'], labels=labels)
cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels],
                     columns=[f'pred_{l}' for l in labels])
print('=== fast-run confusion matrix (rows = truth, cols = prediction) ===')
print(cm_df.to_string())
print('\n=== fast-run per-condition report ===')
print(classification_report(pred['true_condition'], pred['predicted_condition'],
                            labels=labels, zero_division=0, digits=4))
cm_df

## 7. Train the real MentalBERT baseline

Checkpoints, splits, and predictions are written under `Models/` **on Drive**, so they
persist. `--fp16` speeds things up on the GPU. Adjust `--batch-size` up (e.g. 32) if the
GPU has spare memory (T4 ≈ 15 GB).

**Crash-safe / resumable.** `--save-steps 500` writes a checkpoint every 500 steps to
Drive (keeping the 2 most recent), and `--resume` picks up from the latest one. So if
the Colab session drops mid-run, **just re-run this cell** — it continues from the last
checkpoint instead of starting over. (Step-based saving disables best-model-at-end; the
final checkpoint is the last step's weights.)

This is the long one — roughly 2.5–4 h for 3 epochs on a T4. For a quick first pass, add
`--num-epochs 1` (and optionally `--max-length 128`) to get a checkpoint + confusion
matrix in ~1 h, then decide if the full run is worth it.

In [ ]:
!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "$ARTIFACTS_ROOT" \
    --num-epochs 3 --batch-size 16 --fp16 \
    --save-steps 500 --resume

## 8. Predict on the held-out test split

Reads `Models/splits/test.csv` and the trained checkpoint, writes
`Models/predictions/test_predictions.csv`, and prints accuracy + macro-F1.

In [ ]:
!python -m src.modeling.predict --artifacts-root "$ARTIFACTS_ROOT"

In [ ]:
import pandas as pd
pred = pd.read_csv(f'{ARTIFACTS_ROOT}/predictions/test_predictions.csv')
print('rows:', len(pred))
pred.head()

## 9. Per-condition confusion matrix + report

Builds the per-condition confusion matrix and precision/recall/F1 report directly
from `test_predictions.csv`. Only conditions actually present in the held-out test set
are shown (today: `bipolar`, `depression` — the other POC classes have no data yet).
This is the reference-point breakdown every later method (noise model → abstention)
must beat.

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

pred = pd.read_csv(f'{ARTIFACTS_ROOT}/predictions/test_predictions.csv')

# Label set = conditions that actually appear as a true label OR a prediction.
labels = sorted(set(pred['true_condition']) | set(pred['predicted_condition']))

cm = confusion_matrix(pred['true_condition'], pred['predicted_condition'], labels=labels)
cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels],
                     columns=[f'pred_{l}' for l in labels])

print('=== Per-condition confusion matrix (rows = truth, cols = prediction) ===')
print(cm_df.to_string())
print('\n=== Per-condition classification report ===')
print(classification_report(pred['true_condition'], pred['predicted_condition'],
                            labels=labels, zero_division=0, digits=4))

# Persist alongside the predictions so later steps (C1 noise model) can reload it.
cm_out = f'{ARTIFACTS_ROOT}/predictions/confusion_matrix.csv'
cm_df.to_csv(cm_out)
print('Wrote confusion matrix ->', cm_out)
cm_df

## 10. C1 diagnostic — extract MentalBERT pooled embeddings (train split)

First step of the condition-dependent noise diagnostic (`src/noise/`, DECISIONS.md
D-030 / open item 2). This is **inference only** — no training. It loads the saved
checkpoint + tokenizer from `Models/checkpoints/latest` on Drive (so the gated-HF
login in step 4b is **not** needed here), runs each training-split post through the
model, and caches the pooled pre-classification-head representation
(`base_model.pooler_output`) to `Models/embeddings/train/` on Drive as
`features.npy` + `metadata.csv`.

`--output-dtype float16` keeps the artefact ~172 MB (cosine-neighbour ordering is
robust to it; local analysis upcasts to float32). A few minutes on a T4. The next
stage (`src/noise/clusterability.py`) runs **locally** on the cached embeddings — no
GPU needed — so bring `Models/embeddings/train/` back down (Drive Desktop sync or
download) and run `python -m src.noise.clusterability --embeddings-dir <path>`.

In [ ]:
# Extract + cache pooled embeddings for the TRAIN split. Inference only; writes
# Models/embeddings/train/{features.npy, metadata.csv} to Drive. ~172 MB at fp16.
# Point --checkpoint-dir elsewhere if your 3-epoch checkpoint is not at
# <ARTIFACTS_ROOT>/checkpoints/latest.
!python -m src.noise.embeddings \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --split train \
    --output-dtype float16 \
    --batch-size 64

In [ ]:
# Sanity-check the cached embeddings before pulling them down for local analysis.
import numpy as np, pandas as pd

emb_dir = f'{ARTIFACTS_ROOT}/embeddings/train'
feats = np.load(f'{emb_dir}/features.npy')
meta = pd.read_csv(f'{emb_dir}/metadata.csv')

print('shape', feats.shape, '| dtype', feats.dtype,
      '| finite', bool(np.isfinite(feats).all()),
      '| aligned', len(feats) == len(meta))
# Expect ~112k rows, no daic_woz, depression-heavy skew. If counts look like the full
# ~140k or include daic_woz, the wrong split was embedded.
print('\nsource(s):', sorted(meta['source'].unique()))
print(meta['condition'].value_counts())

## 10b. D-035 control — base MentalBERT embeddings

The clusterability diagnostic on the fine-tuned embeddings (§10) came back with high
agreement for every condition, including bipolar (DECISIONS.md **D-034**: the
prediction was falsified). One confound is that those embeddings were fine-tuned to
separate these same noisy labels, so the structure may be a training artifact rather
than intrinsic to the text.

This control re-extracts using **base `mental/mental-bert-base-uncased`** (no
fine-tuning on our labels). The **delta** between fine-tuned and base agreement
quantifies how much of §10's structure is artifact. It does **not** produce a "true"
clusterability number (true labels are unavailable).

**The extract cell below needs the step 4b HF login** (base MentalBERT is gated). It
writes to a distinct Drive path so §10's cache is preserved. The comparison cell after
it runs the diagnostic (torch-free, no GPU) on **both** caches straight from Drive, on
the same protocol as D-034 (seed 0, |E| = 15000, G = 20), and prints the
`delta = finetuned - base` table. You can also run that same command in your local
`.venv` after syncing the two folders down; the result is identical (deterministic).

**Interpretation rule, set in advance (D-035):** base bipolar agreement above ~70%
means the clusterability-failure argument is genuinely weak and should be dropped;
below ~40% means §10's number was substantially artifact. HOC itself (Stage 2) still
runs on the **fine-tuned** embeddings, matching HOC's own protocol, not on these.

In [ ]:
# D-035 control: extract embeddings from BASE MentalBERT (NO fine-tuning on our four
# labels). REQUIRES the HF login in step 4b -- base MentalBERT is a gated repo (the
# fine-tuned run in step 10 did not need it, because it loads a local checkpoint).
# Writes to a DISTINCT path Models/embeddings/train__base/ so the fine-tuned cache is
# never overwritten. ~172 MB at fp16, a few minutes on a T4.
!python -m src.noise.embeddings \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --extractor base \
    --split train \
    --output-dtype float16 \
    --batch-size 64

In [ ]:
# D-035 comparison: run the clusterability diagnostic on BOTH caches (fine-tuned vs
# base) straight from Drive, same protocol as D-034 (seed 0, |E|=15000, G=20).
# Torch-free; prints both reports + the delta table and writes the CSVs to Drive.
# (Identical if run in the local .venv instead.)
!python -m src.noise.clusterability \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train" \
    --base-embeddings-dir "$ARTIFACTS_ROOT/embeddings/train__base" \
    --seed 0 --sample-size 15000 --n-rounds 20